# Evaluation: base vs fine-tuned

Scores the base Qwen2.5-1.5B-Instruct and the QLoRA-tuned version on the same 200 held-out examples. Two metrics:

- **exact match**: generated SQL == reference SQL, character for character
- **normalized match**: same but case-insensitive, whitespace collapsed, trailing `;` stripped. Fairer, since `SELECT Name` vs `select name` is the same query.

Needs a T4 runtime and the adapter saved to Drive by the training notebook. Generation is one example at a time to keep the code dead simple, ~15 min per model.

In [ ]:
!pip install -q -U bitsandbytes datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ADAPTER_DIR = "/content/drive/MyDrive/qwen25-sql-lora"

from datasets import load_dataset

DATA_URL = "https://raw.githubusercontent.com/Akshu24Tech/text2sql-qlora/main/data/eval.jsonl"
eval_ds = load_dataset("json", data_files=DATA_URL, split="train")
rows = list(eval_ds.select(range(200)))
len(rows)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM = "You are a text-to-SQL assistant. Given a table schema and a question, reply with only the SQL query, nothing else."

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def ask(model, row):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

base_preds = [ask(model, r) for r in tqdm(rows)]

In [ ]:
# same model object, adapter loaded on top from Drive
from peft import PeftModel

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
tuned_preds = [ask(model, r) for r in tqdm(rows)]

In [ ]:
import re

def normalize_sql(s):
    s = s.strip().rstrip(";").lower()
    s = re.sub(r"\s+", " ", s)
    return s

def score(preds, refs):
    exact = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
    norm = sum(normalize_sql(p) == normalize_sql(r) for p, r in zip(preds, refs))
    n = len(refs)
    return {"exact_match": round(exact / n, 3), "normalized_match": round(norm / n, 3), "n": n}

refs = [r["answer"] for r in rows]
results = {
    "base": score(base_preds, refs),
    "fine_tuned": score(tuned_preds, refs),
}
results

In [ ]:
# look at a few cases where the tuned model fixed the base model
shown = 0
for r, b, t in zip(rows, base_preds, tuned_preds):
    ref = normalize_sql(r["answer"])
    if normalize_sql(t) == ref and normalize_sql(b) != ref:
        print("Q:", r["question"])
        print("ref  :", r["answer"])
        print("base :", b)
        print("tuned:", t)
        print("-" * 80)
        shown += 1
    if shown == 5:
        break

In [ ]:
# and a few the tuned model still gets wrong, worth discussing in the report
shown = 0
for r, t in zip(rows, tuned_preds):
    if normalize_sql(t) != normalize_sql(r["answer"]):
        print("Q:", r["question"])
        print("ref  :", r["answer"])
        print("tuned:", t)
        print("-" * 80)
        shown += 1
    if shown == 5:
        break

In [ ]:
import json

# saved next to the adapter on Drive, download from there into results/ in the repo
with open(f"{ADAPTER_DIR}/metrics.json", "w") as f:
    json.dump(results, f, indent=2)

with open(f"{ADAPTER_DIR}/predictions.json", "w") as f:
    json.dump(
        [
            {"question": r["question"], "reference": r["answer"], "base": b, "tuned": t}
            for r, b, t in zip(rows, base_preds, tuned_preds)
        ],
        f,
        indent=2,
    )
print("saved to Drive")